In [7]:
from f_0_dirs import get_data_dirs
dirs = get_data_dirs()

# Iterate over the attributes of the DirPaths object
for attr in dir(dirs):
    if attr.startswith('_'):
        continue
    if callable(getattr(dirs, attr)):
        continue
    print(f"{attr}: {getattr(dirs, attr)}")

data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\1_FAME_raw_data\2025.07.30
output_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\output
raw_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\1_FAME_raw_data\2025.02
root_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean
root_dir: C:\Users\lazyst\Files\ucl\Dissertation
work_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\src


# 1. [read] from FAME

### Categorise raw files

This script resolves the project data paths, scans the raw-data folder, builds a nested dictionary of file metadata by company and file category, and writes it to JSON.  
`get_data_dirs()` defines the working directories  
`build_raw_file_dict()` performs the recursive traversal and file collection through its helper functions.  

In [8]:
from flask import json
import pandas as pd

from f_1_traverse import build_raw_file_dict
pd.options.mode.chained_assignment = None  # default='warn'

if dirs.raw_data_dir is None:
	raise ValueError("raw_data_dir is None. Please check your .env file and ensure RAW_DATA_DIR is set correctly.")

raw_file_dict = build_raw_file_dict(dirs.raw_data_dir)
with open(dirs.output_dir / "raw_file_dict.json", "w") as f:
    json.dump(raw_file_dict, f, indent=4)
    print(f"✅ Successfully built raw file dictionary and saved to: {dirs.output_dir / 'raw_file_dict.json'}")

Traversing industry directory: 01, 02, 03, 05, 06, 07, 08, 09, 10, 11, 12, 13, 14, 15, 16
17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31
32, 33, 35, 36, 37, 38, 39, 41, 42, 43, 45, 46, 47, 49, 50
51, 52, 53, 55, 56, 58, 59, 60, 61, 62, 63, 64, 65, 66, 68
69, 70, 71, 72, 73, 74, 75, 77, 78, 79, 80, 81, 82, 84, 85
86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99
✅ Successfully built raw file dictionary and saved to: C:\Users\lazyst\Files\ucl\Dissertation\build\output\raw_file_dict.json


### Process each excel file in the raw file dictionary

In [15]:
import pandas as pd

# import xlsx file from input/raw_properties.xlsx to load as a schema
# Declare types that the schema_source df has colums ["from_raw", "key", "type", "fuzzy_mapping", "in_raw_data", "keep", "in_ln_set", "description"]
schema_path = dirs.root_dir / "build" / "input" / "raw_properties.xlsx"
schema_source = pd.read_excel(schema_path, sheet_name="raw_properties", engine="openpyxl")

# Schema for raw inputs is rows where from_raw is not blank
schema_raw: pd.DataFrame = schema_source[schema_source["from_raw"].notna()]

# Take our master schema and turn it into a helpful mapping of fuzzy column names to schema column names
# Regardless of what the raw source is. We'll filter it later
schema_fixed_fuzzy_df: pd.DataFrame = schema_source[["key", "fuzzy_mapping"]]
schema_fixed_fuzzy_df["fml"] = schema_fixed_fuzzy_df["fuzzy_mapping"].str.split('\n')

# Print any rows where fml has more than one element
for index, row in schema_fixed_fuzzy_df.iterrows():
    if row["fml"] is None:
        print(f"Row {index} has fml that is None: {row['key']}")
    elif type(row["fml"]) is not list:
        print(f"Row {index} has fml that is not a list: {row['key']}")
    elif len(row["fml"]) > 1:
        print(f"Row {index} has more than one fuzzy mapping: {row['fml']}")

Row 37 has more than one fuzzy mapping: ['Strategy,  organization and policy', 'Strategy, organization and policy']
Row 49 has fml that is not a list: has_ptaddress
Row 50 has fml that is not a list: has_ptaddress_latlong
Row 51 has fml that is not a list: is_public
Row 53 has fml that is not a list: industry_code
Row 54 has fml that is not a list: file_code


# 2. [write] To duck schemas

### Define duck schemas

In [16]:
import pandas as pd
# Import ibis-framework
import ibis
# pip install 'ibis-framework[duckdb]'

print("Path:", ibis.__file__)
print("Version:", getattr(ibis, "__version__", "No version found"))
pd.options.mode.chained_assignment = None  # default='warn'

db_path = dirs.output_dir / "fame_data.duckdb"

# Fixed schema
# schema_fixed is a df of schema_source where values in column "keep" are "fixed" or "all"
schema_fixed: pd.DataFrame = schema_source[schema_source["keep"].isin(["fixed", "all"])]
schema_fixed_dict: dict[str, str] = dict(zip(schema_fixed["key"], schema_fixed["type"]))
schema_fixed_ibis: ibis.Schema = ibis.schema(schema_fixed_dict)
schema_fixed_names: set[str] = set(schema_fixed_ibis.keys())

schema_derived: pd.DataFrame = schema_source[schema_source["keep"].isin(["derived", "all"])]
schema_derived_dict: dict[str, str] = dict(zip(schema_derived["key"], schema_derived["type"]))
schema_derived_ibis: ibis.Schema = ibis.schema(schema_derived_dict)
schema_derived_names: set[str] = set(schema_derived_ibis.keys())

# 4. Execute the table creation using the Ibis schema
try:
    
    # 2. Connect to DuckDB using Ibis
    con = ibis.duckdb.connect(str(db_path))
    print(f"Initializing DuckDB via Ibis at: {db_path}")
    # overwrite=True prevents errors if the script is run multiple times during setup
    con.create_table("fame_fixed", schema=schema_fixed_ibis, overwrite=True)
    print("✅ Successfully created Ibis schema for 'fame_fixed'.")
    print("\nTable Schema Verification:")
    print(con.table("fame_fixed").schema())

    con.create_table("fame_derived", schema=schema_derived_ibis, overwrite=True)
    print("✅ Successfully created Ibis schema for 'fame_derived'.")
    print("\nTable Schema Verification:")
    print(con.table("fame_derived").schema())

except Exception as e:
    print(f"❌ Error creating table: {e}")

Path: c:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\Lib\site-packages\ibis\__init__.py
Version: 12.0.0
Initializing DuckDB via Ibis at: C:\Users\lazyst\Files\ucl\Dissertation\build\output\fame_data.duckdb
✅ Successfully created Ibis schema for 'fame_fixed'.

Table Schema Verification:
ibis.Schema {
  company_name                     string
  registered_number                string
  primary_uk_sic_2007_code         string
  primary_uk_sic_2007_description  string
  latest_accounts_date             date
  no_of_available_years            int32
}
✅ Successfully created Ibis schema for 'fame_derived'.

Table Schema Verification:
ibis.Schema {
  registered_number      string
  has_ptaddress          boolean
  has_ptaddress_latlong  boolean
  is_public              boolean
  industry_code          string
  file_code              string
}


### Load, modify and write imported file schema

In [ ]:
from flask import json
from f_1_traverse import RawFileDict
from f_2_check import drop_duplicate_columns, check_df_matches_schema, set_na_columns, fix_excel_dates
from f_2_modify import set_data_av_flags
import random

# Traverse the raw_file_dict.json file to get each Excel filepath
# declare raw_file_dict as a RawFileDict type
raw_file_dict: RawFileDict | None = None
with open(dirs.output_dir / "raw_file_dict.json", "r") as f:
    raw_file_dict = json.load(f)
if raw_file_dict is None:
    raise ValueError("❌ Error: raw_file_dict.json is empty or not found.")
if dirs.raw_data_dir is None:
    raise ValueError("❌ Error: raw_data_dir is None. Please check your .env file and ensure RAW_DATA_DIR is set correctly.")

process_count = 0

# We want one big dataframe, which we will merge all the data into for now
# df_fixed = pd.DataFrame(columns=list(schema_fixed.columns))
ind_keys = raw_file_dict.keys()
ind_shuffled = list(ind_keys)
random.shuffle(ind_shuffled)

for ind in ind_shuffled:

    obj = raw_file_dict[ind]

    for property, arr in obj.items():

        if property != "a1_ID":
            continue

        # DECLARE THE SCHEMA for this input
        schema_raw_fuzzy_filtered = schema_raw[schema_raw["from_raw"].isin([property, 'all'])]
        schema_raw_fuzzy_filtered["fml"] = schema_raw_fuzzy_filtered["fuzzy_mapping"].str.split('\n')
        schema_raw_fuzzy_mapping: dict[str, str] = dict(zip(
            schema_raw_fuzzy_filtered["key"],
            schema_raw_fuzzy_filtered["fml"]
        ))
        schema_raw_fuzzy_col_map = {
            raw_name: schema_name for schema_name,
            raw_names in schema_raw_fuzzy_mapping.items() for raw_name in raw_names
        }

        files_shuffled = arr.copy()
        random.shuffle(files_shuffled)
        for [file_name, file_path] in files_shuffled:
            if process_count >= 5:
                break
            # Export 22_01_2026 09_39 1.xlsx
            # file_code is split the file_name by spaces
            # and then drop the first 2 elements, join the rest with spaces, and then drop the .xlsx extension
            file_code = " ".join(file_name.split()[2:]).replace(".xlsx", "")

            try:

                # LOAD
                df_raw = pd.read_excel(file_path, sheet_name='Results', header=0)       # Read Excel file
                df_raw.drop(df_raw.columns[0], axis=1, inplace=True)                    # Drop column A (blank in raw data)
                df_raw = drop_duplicate_columns(df_raw)                                 # Remove duplicate columns (quirk of some files)
                df_raw.rename(columns=schema_raw_fuzzy_col_map, inplace=True)           # Rename according to our mapping

                # FILTER
                df_raw = df_raw[df_raw['registered_number'].notna()]                    # Drop rows with NaN registered number
                df_raw = df_raw[df_raw['no_of_available_years'] != 0]                   # Drop rows with 0 in 'no_of_available_years'
                df_raw = fix_excel_dates(df_raw)                                        # Fix any Excel date serials to datetime
                try:
                    check_df_matches_schema(schema_raw_fuzzy_mapping, df_raw)               # Check if the DataFrame matches the input
                except ValueError as e:
                    print(f"⚠️ Mismatch in raw data validation for file {ind}/{property}/{file_name})")
                    print("Warning:", e)
                    
                # MODIFY
                df_fixed_one = df_raw[[col for col in df_raw.columns if col in schema_fixed_names]]
                set_na_columns(schema_fixed_names, df_fixed_one)  # Add any missing columns with NaN values
                check_df_matches_schema(schema_fixed_ibis, df_fixed_one)

                # DERIVE
                df_derived = pd.DataFrame(columns=list(schema_derived_names))
                df_derived['registered_number'] = df_raw['registered_number']
                df_derived['industry_code'] = ind
                # file
                df_derived['file_code'] = file_code
                df_derived_flagged = set_data_av_flags(df_raw, df_derived)
                df_derived_fullCol = set_na_columns(schema_derived_names, df_derived_flagged)
                check_df_matches_schema(schema_derived_ibis, df_derived_fullCol)

                # WRITE
                # # Insert data into DuckDB table
                con.insert("fame_fixed", df_fixed_one)
                print(f"✅ Successfully processed fixed table from: {ind}/{property}/{file_name}")

                con.insert("fame_derived", df_derived_fullCol)
                print(f"✅ Successfully processed derived table from: {ind}/{property}/{file_name}")
                
            except Exception as e:
                print(f"❌ Error processing file {ind}/{property}/{file_name}")
                print("Error:", e)
                print("")

            process_count += 1

# Output the head of the fame_fixed and fame_derived tables to verify the data
print("\nHead of fame_fixed table:")
print(con.table("fame_fixed").execute().head())
print("\nHead of fame_derived table:")
print(con.table("fame_derived").execute().head())

✅ Successfully processed fixed table from: 56/a1_ID/Export 19_02_2025 17_31 2.xlsx
✅ Successfully processed derived table from: 56/a1_ID/Export 19_02_2025 17_31 2.xlsx
⚠️ Converted Excel date serials to datetime for column 'latest_accounts_date'
✅ Successfully processed fixed table from: 56/a1_ID/Export 19_02_2025 17_27 2.xlsx
✅ Successfully processed derived table from: 56/a1_ID/Export 19_02_2025 17_27 2.xlsx
⚠️ Converted Excel date serials to datetime for column 'latest_accounts_date'
✅ Successfully processed fixed table from: 56/a1_ID/Export 19_02_2025 17_30 1.xlsx
✅ Successfully processed derived table from: 56/a1_ID/Export 19_02_2025 17_30 1.xlsx
⚠️ Converted Excel date serials to datetime for column 'latest_accounts_date'
✅ Successfully processed fixed table from: 56/a1_ID/Export 19_02_2025 17_26 1.xlsx
✅ Successfully processed derived table from: 56/a1_ID/Export 19_02_2025 17_26 1.xlsx
⚠️ Converted Excel date serials to datetime for column 'latest_accounts_date'
✅ Successfully p

# 3. [view] resulting DB for inspection

### basic tables overview

In [18]:
# List tables in the DuckDB database as an .md file in /tmp
# Give me the head of all tables
# Ensure they are nicely formatted with headers so I can easily see what's going on
out_file = dirs.root_dir / "build" / "tmp" / "duckdb_tables.md"
with open(out_file, "w") as f:
    tables = con.list_tables()
    f.write("# Tables in DuckDB database\n\n")
    for table in tables:
        f.write(f"## {table}\n\n")
        f.write(f"### Number of rows: {con.table(table).count().execute()}\n\n")
        f.write(f"### Schema:\n\n```\n{con.table(table).schema()}\n```\n\n")
        f.write(f"### Head of table:\n\n```\n{con.table(table).execute().head()}\n```\n\n")
print(f"✅ Successfully listed tables and their heads in: {out_file}")

✅ Successfully listed tables and their heads in: C:\Users\lazyst\Files\ucl\Dissertation\build\tmp\duckdb_tables.md


In [ ]:
# Dump the first 500 rows of df_raw to a CSV file in /tmp for inspection
out_file_raw = dirs.root_dir / "build" / "tmp" / "df_raw_head.csv"
df_raw.head(500).to_csv(out_file_raw, index=False)
print(f"✅ Successfully dumped the first 500 rows of df_raw to: {out_file_raw}")

### Start joining tables

In [ ]:
# Select from duckdb where any company doesn't have a registered number
con = ibis.duckdb.connect(str(db_path))
query = con.table("fame_id_data").filter(con.table("fame_id_data").registered_number.isnull() | (con.table("fame_id_data").registered_number == ""))
for row in query.execute().to_dict(orient="records"):
    print(row)
# Bind Ibis to the existing DuckDB tables
fixed = con.table("fame_fixed")
panel = con.table("fame_panel")
derived = con.table("fame_derived")

# Construct a lazy relational join using the registered_number key
master_query = (
    panel
    .left_join(fixed, "registered_number")
    .left_join(derived, "registered_number")
    # Explicitly select only the variables required for the current regression/analysis
    .select([
        panel.registered_number,
        panel.year,
        panel.turnover_gbp,
        fixed.primary_uk_sic_2007_code,
        derived.distance_to_parent_km
    ])
    # Apply global filters before execution to minimise memory load
    .filter(derived.active_subsidiary_flag == True)
)

# Execute the query in C++ and pull the final structured panel to Pandas
df_analysis = master_query.execute()